# Week 1 — Linear Regression 1: Three Marketing Datasets
**DX799 Capstone | Cindy Lopez**

**Research question:** Do polynomial and interaction terms improve out-of-sample prediction of marketing outcomes relative to linear regression, and what does multicollinearity reveal about the features?

Three independent datasets are analyzed **separately** because their rows, outcomes, and definitions differ. This is a predictive comparison, not a causal analysis. All metrics are computed from held-out observations. The primary dataset predicts continuous `ConversionRate`; the two additional datasets predict campaign-level `Conversions` counts. These metrics must not be compared directly across datasets.

**Data provenance:** El Kharoua (2024), *Predict conversion in digital marketing*; Nayak (2024), *Google Ads sales dataset*; Shah (2024), *Marketing and product performance dataset* (Kaggle).

In [1]:
import numpy as np, pandas as pd, os, warnings
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from IPython.display import display

FILES = {
 'Customer conversion': 'digital_marketing_campaign_dataset.csv',
 'Google Ads': 'GoogleAds_DataAnalytics_Sales_Uncleaned.csv',
 'Product performance': 'marketing_and_product_performance.csv'
}
for label, filename in FILES.items():
    if not os.path.isfile(filename):
        raise FileNotFoundError(f'Missing {filename}. Place all three CSVs in the notebook folder or upload them to Colab.')
print('All three CSV files found.')

All three CSV files found.


## 1. EDA and data preparation
The Google Ads CSV is explicitly uncleaned. Currency symbols and commas are removed before numeric conversion. Missing predictors are imputed using training-set statistics inside the pipeline. Rows without an observed outcome are excluded. Identifiers, target proxies, and post-outcome fields are not used as predictors. In the primary dataset, `Conversion` is excluded when predicting `ConversionRate`; for the other datasets, revenue, ROI, and sale amount are excluded because they can reflect events occurring after conversion.

In [2]:
data={name:pd.read_csv(filename) for name,filename in FILES.items()}
for name, df in data.items():
    print('\n',name,'shape:',df.shape,'missing cells:',int(df.isna().sum().sum()),'duplicates:',int(df.duplicated().sum()))
    display(df.head(2))
    display(df.isna().sum().sort_values(ascending=False).head(5).to_frame('missing'))


 Customer conversion shape: (8000, 20) missing cells: 0 duplicates: 0


,CustomerID,Age,Gender,Income,CampaignChannel,CampaignType,AdSpend,ClickThroughRate,ConversionRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,EmailClicks,PreviousPurchases,LoyaltyPoints,AdvertisingPlatform,AdvertisingTool,Conversion
0,8000,56,Female,136912,Social Media,Awareness,6497.870068,0.043919,0.088031,0,2.399017,7.396803,19,6,9,4,688,IsConfid,ToolConfid,1
1,8001,69,Male,41760,Email,Retention,3898.668606,0.155725,0.182725,42,2.917138,5.352549,5,2,7,2,3459,IsConfid,ToolConfid,1


,missing
CustomerID,0
Age,0
Gender,0
Income,0
CampaignChannel,0



 Google Ads shape: (2600, 13) missing cells: 1150 duplicates: 0


,Ad_ID,Campaign_Name,Clicks,Impressions,Cost,Leads,Conversions,Conversion Rate,Sale_Amount,Ad_Date,Location,Device,Keyword
0,A1000,DataAnalyticsCourse,104.0,4498.0,$231.88,14.0,7.0,0.058,$1892,2024-11-16,hyderabad,desktop,learn data analytics
1,A1001,DataAnalyticsCourse,173.0,5107.0,$216.84,10.0,8.0,0.046,$1679,20-11-2024,hyderabad,mobile,data analytics course


,missing
Conversion Rate,626
Sale_Amount,139
Clicks,112
Cost,97
Conversions,74



 Product performance shape: (10000, 17) missing cells: 0 duplicates: 0


,Campaign_ID,Product_ID,Budget,Clicks,Conversions,Revenue_Generated,ROI,Customer_ID,Subscription_Tier,Subscription_Length,Flash_Sale_ID,Discount_Level,Units_Sold,Bundle_ID,Bundle_Price,Customer_Satisfaction_Post_Refund,Common_Keywords
0,CMP_RLSDVN,PROD_HBJFA3,41770.45,4946,73,15520.09,1.94,CUST_1K7G39,Premium,4,FLASH_1VFK5K,43,34,BNDL_29U6W5,433.80,4,Affordable
1,CMP_JHHUE9,PROD_OE8YNJ,29900.93,570,510,30866.17,0.76,CUST_0DWS6F,Premium,4,FLASH_1M6COK,28,97,BNDL_ULV60J,289.29,2,Innovative


,missing
Campaign_ID,0
Product_ID,0
Budget,0
Clicks,0
Conversions,0


In [3]:
ads=data['Google Ads'].copy()
for col in ['Cost','Sale_Amount']:
    ads[col]=pd.to_numeric(ads[col].astype(str).str.replace(r'[$,]','',regex=True),errors='coerce')
for col in ['Clicks','Impressions','Leads','Conversions','Conversion Rate']:
    ads[col]=pd.to_numeric(ads[col],errors='coerce')
# We deliberately exclude Leads, Conversion Rate and Sale_Amount as outcome-adjacent or downstream variables.
config={
 'Customer conversion':{'frame':data['Customer conversion'],'target':'ConversionRate','numeric':['Age','Income','AdSpend','ClickThroughRate','WebsiteVisits','PagesPerVisit','TimeOnSite','SocialShares','EmailOpens','EmailClicks','PreviousPurchases','LoyaltyPoints'],'categorical':['Gender','CampaignChannel','CampaignType'],'poly':['AdSpend','ClickThroughRate','TimeOnSite']},
 'Google Ads':{'frame':ads,'target':'Conversions','numeric':['Clicks','Impressions','Cost'],'categorical':['Campaign_Name','Location','Device'],'poly':['Clicks','Impressions','Cost']},
 'Product performance':{'frame':data['Product performance'],'target':'Conversions','numeric':['Budget','Clicks','Discount_Level','Bundle_Price'],'categorical':['Subscription_Tier','Common_Keywords'],'poly':['Budget','Clicks','Discount_Level']}
}
for name,c in config.items():
    print(name,'target:',c['target'],'range:',c['frame'][c['target']].min(),c['frame'][c['target']].max())

Customer conversion target: ConversionRate range: 0.0100177828330717 0.1999947079999166
Google Ads target: Conversions range: 3.0 10.0
Product performance target: Conversions range: 1 999


## 2. Baseline versus polynomial/interaction regression
We use an 80/20 random train/test split with seed 42. The baseline includes scaled numeric predictors and one-hot encoded categorical predictors. The polynomial model adds squared and pairwise interaction terms for **three prespecified numeric predictors** and uses Ridge regularization to constrain the increased model complexity. We compare test R², RMSE and MAE. Ridge alpha=10 is an illustrative fixed setting, **not a tuned optimum**. This is a Week 1 exploratory comparison, not evidence of causation.

In [4]:
def make_encoder():
    try:return OneHotEncoder(handle_unknown='ignore',sparse_output=False)
    except TypeError:return OneHotEncoder(handle_unknown='ignore',sparse=False)

def evaluate(name,c):
    df=c['frame']; target=c['target']; cols=c['numeric']+c['categorical']
    work=df[cols+[target]].copy()
    work[target]=pd.to_numeric(work[target],errors='coerce')
    work=work.dropna(subset=[target])
    X,y=work[cols],work[target]
    xtr,xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42)
    numeric_pipe=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())])
    cat_pipe=Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',make_encoder())])
    baseline=Pipeline([('prep',ColumnTransformer([('num',numeric_pipe,c['numeric']),('cat',cat_pipe,c['categorical'])])),('model',LinearRegression())])
    poly_pipe=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('poly',PolynomialFeatures(degree=2,include_bias=False))])
    other=[v for v in c['numeric'] if v not in c['poly']]
    poly=Pipeline([('prep',ColumnTransformer([('poly',poly_pipe,c['poly']),('num',numeric_pipe,other),('cat',cat_pipe,c['categorical'])])),('model',Ridge(alpha=10))])
    out=[]
    for label,model in [('Linear baseline',baseline),('Polynomial + interactions (Ridge)',poly)]:
        model.fit(xtr,ytr); pred=model.predict(xte)
        out.append({'dataset':name,'model':label,'n_train':len(xtr),'n_test':len(xte),'R2':r2_score(yte,pred),'RMSE':np.sqrt(mean_squared_error(yte,pred)),'MAE':mean_absolute_error(yte,pred)})
    return out
results=pd.DataFrame([row for name,c in config.items() for row in evaluate(name,c)])
display(results.round(4))

,dataset,model,n_train,n_test,R2,RMSE,MAE
0,Customer conversion,Linear baseline,6400,1600,-0.0023,0.0541,0.0469
1,Customer conversion,Polynomial + interactions (Ridge),6400,1600,-0.0057,0.0542,0.0470
2,Google Ads,Linear baseline,2020,506,0.0007,2.2547,1.9564
3,Google Ads,Polynomial + interactions (Ridge),2020,506,-0.0039,2.2599,1.9575
4,Product performance,Linear baseline,8000,2000,-0.0007,291.0550,252.3043
5,Product performance,Polynomial + interactions (Ridge),8000,2000,-0.0027,291.3538,252.4118


## 3. Multicollinearity: variance inflation factors
VIF measures how much the variance of a regression coefficient is inflated by correlation with other predictors. Values greater than 5 can warrant investigation, but are not an automatic exclusion rule. We calculate VIF on the **numeric baseline predictors in training data only**, after median imputation. One-hot categorical variables are omitted from this diagnostic to avoid coding artifacts.

In [5]:
vif_rows=[]
for name,c in config.items():
    frame=c['frame'].dropna(subset=[c['target']]); xtr,_=train_test_split(frame[c['numeric']],test_size=.2,random_state=42)
    x=SimpleImputer(strategy='median').fit_transform(xtr)
    x=StandardScaler().fit_transform(x)
    for j,col in enumerate(c['numeric']):
        try:v=float(variance_inflation_factor(x,j))
        except Exception:v=float('nan')
        vif_rows.append({'dataset':name,'feature':col,'VIF':v})
vifs=pd.DataFrame(vif_rows)
display(vifs.sort_values('VIF',ascending=False).groupby('dataset').head(6).round(3))

,dataset,feature,VIF
6,Customer conversion,TimeOnSite,1.002
4,Customer conversion,WebsiteVisits,1.002
5,Customer conversion,PagesPerVisit,1.002
11,Customer conversion,LoyaltyPoints,1.002
2,Customer conversion,AdSpend,1.002
7,Customer conversion,SocialShares,1.002
14,Google Ads,Cost,1.001
12,Google Ads,Clicks,1.001
17,Product performance,Discount_Level,1.000
16,Product performance,Clicks,1.000


## 4. Interpretation and limitations
The comparison above answers whether the added complexity improves *held-out* predictive performance in each dataset. A negative R² means the model performs worse than predicting the test-set mean. RMSE and MAE are in the target's units; they cannot be compared across customer conversion rates and campaign conversion counts. Categorical variables are one-hot encoded rather than treated as numeric ranks. Polynomial terms can increase variance and collinearity, which is why we also evaluate a regularized model.

**Cautions:** No independent external validation was performed. We have not tuned polynomial degree or Ridge alpha in this exploratory notebook. The Google Ads dataset contains missing data; median/mode imputation is a pragmatic baseline, not proof that missingness is random. The datasets may be synthetic or constructed and may not represent live campaign performance. Report associations and predictions, not causal effects.

**For Milestone One:** Cite the numerical output above, discuss whether each polynomial model improved test RMSE/R², and use the VIF table to explain whether multicollinearity appears substantial. Do not copy any model conclusion that is not supported by the executed output.

## References and AI-use note
- El Kharoua, R. (2024). *Predict conversion in digital marketing dataset*. Kaggle. https://www.kaggle.com/datasets/rabieelkharoua/predict-conversion-in-digital-marketing-dataset
- Nayak, G. (2024). *Google Ads sales dataset*. Kaggle. https://www.kaggle.com/datasets/nayakganesh007/google-ads-sales-dataset
- Shah, I. A. (2024). *Marketing and product performance dataset*. Kaggle. https://www.kaggle.com/datasets/imranalishahh/marketing-and-product-performance-dataset

**AI assistance:** ChatGPT was used to draft and organize notebook code, preprocessing steps, and methodological explanations in response to a request to apply the Week 1 techniques to three digital marketing datasets. All displayed numerical results are produced by the executed Python cells and should be independently reviewed before submission.